# GenAI: A RAG Analytics Assistant (POC)

**Author:** Aman Bohra · Senior Analytics & BI Professional
**Portfolio:** https://amandbohra.github.io/portfolio/

A common enterprise ask: *"let business users ask questions of our analytics docs in plain English."*
This notebook builds a lightweight **Retrieval-Augmented Generation (RAG)** pipeline — the retrieval
half runs here for real (TF-IDF over a small knowledge base); the generation step is shown as a clear
plug-in point for an LLM (Claude / OpenAI). It reflects my Databricks GenAI Engineer certification and
how I'd frame a governed GenAI POC — grounded answers, no hallucinated numbers.

> ⚠️ Illustrative POC on synthetic docs. The LLM call is stubbed so the notebook runs without API keys.


In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 1. Knowledge base
A few short "analytics glossary / KPI definition" snippets — the governed source of truth.

In [2]:
KB = [
 {"id":"kpi-written-premium","text":"Written Premium is the total premium on policies issued during a period, before deductions. It measures new business volume."},
 {"id":"kpi-earned-premium","text":"Earned Premium is the portion of written premium that corresponds to coverage already provided in the period. Earned = Written adjusted for the unexpired portion."},
 {"id":"kpi-loss-ratio","text":"Loss Ratio is incurred losses divided by earned premium. A lower loss ratio indicates better underwriting profitability."},
 {"id":"kpi-combined-ratio","text":"Combined Ratio is the loss ratio plus the expense ratio. Below 100% means an underwriting profit; above 100% means a loss."},
 {"id":"gov-maker-checker","text":"Maker-checker is a governance control where one person prepares a change and a second reviews and approves it before it reaches production, reducing errors."},
 {"id":"proc-refresh","text":"The distributor performance dashboard refreshes daily at 6am from the warehouse and covers all international regions."},
]
docs = [d["text"] for d in KB]
len(KB)


6

## 2. Retrieval — embed & search (runs for real)
TF-IDF vectors + cosine similarity retrieve the most relevant snippets for a question.

In [3]:
vec = TfidfVectorizer(stop_words="english")
M = vec.fit_transform(docs)

def retrieve(query, k=2):
    qv = vec.transform([query])
    sims = cosine_similarity(qv, M)[0]
    idx = sims.argsort()[::-1][:k]
    return [(KB[i]["id"], docs[i], float(sims[i])) for i in idx if sims[i] > 0.05]

for hid, txt, score in retrieve("How is loss ratio calculated?"):
    print(f"[{score:.2f}] {hid}: {txt}")


[0.63] kpi-combined-ratio: Combined Ratio is the loss ratio plus the expense ratio. Below 100% means an underwriting profit; above 100% means a loss.
[0.62] kpi-loss-ratio: Loss Ratio is incurred losses divided by earned premium. A lower loss ratio indicates better underwriting profitability.


## 3. Generation — where the LLM plugs in
The retrieved context is injected into a prompt. In production this calls Claude/OpenAI; here it's a
transparent stub so the notebook runs offline. Note the guardrail: **answer only from context.**

In [4]:
def build_prompt(question, context):
    ctx = "\n".join(f"- {c}" for c in context)
    return (f"You are an analytics assistant. Answer ONLY from the context. "
            f"If the answer isn't in the context, say you don't know.\n\n"
            f"Context:\n{ctx}\n\nQuestion: {question}\nAnswer:")

def call_llm(prompt):
    # ---- Plug in your model here, e.g.: ----
    # from anthropic import Anthropic
    # return Anthropic().messages.create(model="claude-...", max_tokens=300,
    #        messages=[{"role":"user","content":prompt}]).content[0].text
    # Stubbed (extractive) fallback so this runs without keys:
    ctx_line = prompt.split("Context:\n")[1].split("\n\nQuestion")[0]
    return "Based on the retrieved definitions:\n" + ctx_line

def answer(question, k=2):
    hits = retrieve(question, k)
    if not hits:
        return "I don't know — that isn't covered in the knowledge base."
    context = [t for _, t, _ in hits]
    return call_llm(build_prompt(question, context))

print(answer("What is the combined ratio and what does it mean?"))
print("\n---\n")
print(answer("When does the distributor dashboard refresh?"))
print("\n---\n")
print(answer("What was our Q3 revenue?"))   # not in KB -> honest 'don't know'


Based on the retrieved definitions:
- Combined Ratio is the loss ratio plus the expense ratio. Below 100% means an underwriting profit; above 100% means a loss.
- Loss Ratio is incurred losses divided by earned premium. A lower loss ratio indicates better underwriting profitability.

---

Based on the retrieved definitions:
- The distributor performance dashboard refreshes daily at 6am from the warehouse and covers all international regions.

---

I don't know — that isn't covered in the knowledge base.


## 4. Why frame it this way
- **Grounded** — answers come from a governed KB, so the model can't invent KPI numbers.
- **Honest failure** — if the context doesn't hold the answer, it says so (critical in regulated domains).
- **Swappable** — retrieval is real and measurable; the LLM is one clearly-isolated call you can harden,
  cache, and cost-control.

This is how I'd pilot GenAI in an enterprise analytics setting: useful, auditable, and safe by design —
not a chatbot bolted onto sensitive data.

*— Aman Bohra*
